In [1]:
import pygame
import sys
import random
import datetime
import os
import csv
from collections import deque

# 尝试导入 pandas
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# ==========================================
# 配置与常量定义
# ==========================================
class Config:
    SCREEN_WIDTH = 1600
    SCREEN_HEIGHT = 900
    TITLE = "Project: Minecraft RL (Auto-Logic + Seq IDs)"
    
    GRID_SIZE = 30  
    ROOM_GRID_WIDTH = 3
    ROOM_GRID_HEIGHT = 3
    
    BACKPACK_CAPACITY = 4
    
    # RL Rewards
    REWARD_STEP = -0.1          
    REWARD_INVALID_MOVE = -0.5  
    REWARD_COLLECT = 0.5        
    REWARD_DITCH = -0.2         
    REWARD_TRANSFORM = 1.0      
    REWARD_MERGE = 2.0          
    REWARD_WIN = 10.0           
    
    # 颜色定义
    COLOR_BG = (46, 52, 64)
    COLOR_PANEL = (59, 66, 82)
    COLOR_BORDER = (76, 86, 106)
    COLOR_TEXT_MAIN = (236, 239, 244)
    COLOR_TEXT_DIM = (216, 222, 233)
    COLOR_ACCENT = (136, 192, 208)
    COLOR_SUCCESS = (163, 190, 140)
    COLOR_HIGHLIGHT = (94, 129, 172)
    COLOR_PLAYER = (255, 80, 80)
    
    # 房间配色
    COLOR_ROOM_BG = (67, 76, 94)      
    COLOR_ROOM_WALL = (200, 210, 220) 
    COLOR_ROOM_VISITED = (70, 80, 100)
    
    FRAGMENT_COLORS = {
        'e1': (255, 100, 100), 'e2': (100, 200, 100), 'e3': (100, 100, 255),
        'e4': (255, 255, 100), 'e5': (255, 100, 255), 'e6': (100, 255, 255),
        'b1': (205, 170, 125), 'b2': (150, 200, 150), 
        'c1': (150, 150, 200), 'c2': (220, 220, 150),
        'd1': (200, 150, 200), 'd2': (150, 200, 200),
        'B': (140, 140, 140), 'C': (180, 180, 180), 'D': (220, 220, 220),
        'a1': (255, 160, 80), 'a2': (60, 160, 255),
        'A': (255, 215, 0)
    }

# 初始化 Pygame
pygame.init()
screen = pygame.display.set_mode((Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT))
pygame.display.set_caption(Config.TITLE)

def get_font(size):
    try: return pygame.font.SysFont('simhei', size)
    except: return pygame.font.SysFont('arial', size)

font_sm = get_font(16)
font_md = get_font(20)
font_lg = get_font(28)
font_xl = get_font(40)

# ==========================================
# 核心逻辑
# ==========================================

RECIPES_AND = {
    'b1': ['e1', 'e2'], 'b2': ['e2', 'e3'],
    'c1': ['e1', 'e3'], 'c2': ['e4', 'e6'],
    'd1': ['e4', 'e5'], 'd2': ['e5', 'e6'],
    'a1': ['B', 'C'],   'a2': ['C', 'D'],
}

RECIPES_OR = {
    'b1': 'B', 'b2': 'B',
    'c1': 'C', 'c2': 'C',
    'd1': 'D', 'd2': 'D',
    'a1': 'A', 'a2': 'A'
}

class RLDataRecorder:
    def __init__(self, participant_id="RL_Agent_01"):
        self.participant_id = participant_id
        self.timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.data_dir = f"rl_data/{self.timestamp_str}"
        os.makedirs(self.data_dir, exist_ok=True)
        self.memory_buffer = [] 
        self.episode_count = 0
        self.step_count = 0
        print(f"Data Recorder Initialized. Output directory: {self.data_dir}")

    def start_episode(self):
        self.episode_count += 1
        self.step_count = 0
    
    def log_action(self, episode, step, map_type, current_room_id, pos_x, pos_y, 
                   backpack, action_name, action_detail, reward, is_valid, game_over):
        self.step_count += 1
        record = {
            "Participant": self.participant_id,
            "Episode_ID": episode,
            "Step_Index": step,
            "Timestamp": datetime.datetime.now().isoformat(),
            "Map_Structure": map_type,
            "Current_Room_ID": current_room_id, # 这里现在记录的是 1, 2, 3...
            "Grid_X": pos_x,
            "Grid_Y": pos_y,
            "Backpack_Content": str(backpack),
            "Backpack_Count": len(backpack),
            "Action_Type": action_name,
            "Action_Detail": str(action_detail),
            "Action_Valid": is_valid,
            "Reward": reward,
            "Game_Over": game_over
        }
        self.memory_buffer.append(record)

    def save_to_file(self):
        if not self.memory_buffer: return
        filename_base = f"{self.data_dir}/game_log_{self.participant_id}"
        if HAS_PANDAS:
            try:
                pd.DataFrame(self.memory_buffer).to_excel(f"{filename_base}.xlsx", index=False)
                print(f"Data saved to Excel: {filename_base}.xlsx")
                return
            except Exception as e: print(f"Excel error: {e}")
        
        try:
            keys = self.memory_buffer[0].keys()
            with open(f"{filename_base}.csv", 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=keys)
                writer.writeheader()
                writer.writerows(self.memory_buffer)
            print(f"Data saved to CSV: {filename_base}.csv")
        except Exception as e: print(f"CSV error: {e}")

class Room:
    def __init__(self, id, logical_pos):
        self.id = id           # 原始坐标ID (e.g., 201)
        self.seq_id = 0        # 新增：顺序ID (e.g., 1, 2, 3...)
        self.logical_pos = logical_pos 
        self.fragment = None 
        self.doors = {}
        self.visited = False
    
    def add_door(self, direction, target_id):
        self.doors[direction] = target_id

class Game:
    def __init__(self, recorder, map_type="Barbell"):
        self.recorder = recorder
        self.map_type = map_type
        self.rooms = {}
        self.player_x = 0
        self.player_y = 0
        self.backpack = []
        self.global_counter = 0
        self.game_over = False
        self.win_reason = ""
        self.logs = deque(maxlen=8)
        self.selected_items = []
        
        self.direction_map = {"north": "Up", "south": "Down", "west": "Left", "east": "Right"}
        self.setup_level(map_type)
        self.recorder.start_episode()
        self.add_log(f"Map: {map_type} (Seq IDs)")

    def add_log(self, text):
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        self.logs.append(f"[{timestamp}] {text}")

    def get_room_by_grid(self, gx, gy):
        lx, ly = gx // 3, gy // 3
        rid = ly * 100 + lx
        return self.rooms.get(rid)

    # [修改] 现在返回顺序ID用于显示和记录
    def get_current_room_seq_id(self):
        r = self.get_room_by_grid(self.player_x, self.player_y)
        return r.seq_id if r else -1

    def _log_step(self, action_name, action_detail, reward, is_valid):
        # 使用 seq_id 记录
        self.recorder.log_action(
            self.recorder.episode_count, self.global_counter, self.map_type,
            self.get_current_room_seq_id(), self.player_x, self.player_y,
            self.backpack.copy(), action_name, action_detail, reward, is_valid, self.game_over
        )

    def _auto_transform(self):
        changed = False
        for i in range(len(self.backpack)):
            item = self.backpack[i]
            if item in RECIPES_OR:
                new_item = RECIPES_OR[item]
                self.backpack[i] = new_item
                self.add_log(f"自动转化: {item} -> {new_item}")
                self.global_counter += 1 
                self._log_step("auto_merge", f"{item}->{new_item}", Config.REWARD_TRANSFORM, True)
                changed = True
        
        if changed:
            self.check_win()
            self._auto_transform()

    # =========================================================
    # 地图生成
    # =========================================================
# =========================================================
    # 地图生成 (Updated based on Visual Designs)
    # =========================================================
# =========================================================
    # 地图生成 (Final Version)
    # =========================================================
    def setup_level(self, map_type):
        self.rooms = {}
        self.map_type = map_type
        start_rid = 0
        
        # 1. 生成特定地图
        if map_type == "Barbell": start_rid = self._setup_barbell()
        elif map_type == "Grid": start_rid = self._setup_grid()
        elif map_type == "Path": start_rid = self._setup_path()
        elif map_type == "Ladder": start_rid = self._setup_ladder()
        
        # 2. 重新分配顺序ID (1, 2, 3...) 用于界面显示
        sorted_keys = sorted(self.rooms.keys())
        for idx, key in enumerate(sorted_keys):
            self.rooms[key].seq_id = idx + 1

        # 3. 设置玩家初始位置
        if start_rid in self.rooms:
            r = self.rooms[start_rid]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        elif self.rooms:
            # 容错：默认放在第一个房间
            r = self.rooms[sorted_keys[0]]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        else:
            self.player_x, self.player_y = 0, 0

    def _create_room(self, x, y):
        rid = y * 100 + x 
        if rid not in self.rooms: self.rooms[rid] = Room(rid, (x, y))
        return rid

    def _connect(self, rid1, rid2, direction):
        rev = {"north":"south", "south":"north", "east":"west", "west":"east"}[direction]
        if rid1 in self.rooms and rid2 in self.rooms:
            self.rooms[rid1].add_door(direction, rid2)
            self.rooms[rid2].add_door(rev, rid1)

    # 辅助函数：断开连接（制造墙壁）
    def _disconnect(self, x, y, direction):
        rid = y * 100 + x
        if rid in self.rooms:
            self.rooms[rid].doors.pop(direction, None)
            # 处理反向连接
            dx, dy = {'north':(0,-1), 'south':(0,1), 'east':(1,0), 'west':(-1,0)}[direction]
            rev = {'north':'south', 'south':'north', 'east':'west', 'west':'east'}[direction]
            rid2 = (y + dy) * 100 + (x + dx)
            if rid2 in self.rooms:
                self.rooms[rid2].doors.pop(rev, None)

    def _place_fragments(self, placement_dict):
        for (x, y), item in placement_dict.items():
            rid = y * 100 + x
            if rid in self.rooms: self.rooms[rid].fragment = item

    # --- 1. Barbell Map (Final V2) ---
# =========================================================
    # 地图生成 (Final Version Updated)
    # =========================================================
# =========================================================
    # 地图生成 (Barbell Optimized V2)
    # =========================================================
    def setup_level(self, map_type):
        self.rooms = {}
        self.map_type = map_type
        start_rid = 0
        
        if map_type == "Barbell": start_rid = self._setup_barbell()
        elif map_type == "Grid": start_rid = self._setup_grid()
        elif map_type == "Path": start_rid = self._setup_path()
        elif map_type == "Ladder": start_rid = self._setup_ladder()
        
        sorted_keys = sorted(self.rooms.keys())
        for idx, key in enumerate(sorted_keys):
            self.rooms[key].seq_id = idx + 1

        if start_rid in self.rooms:
            r = self.rooms[start_rid]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        elif self.rooms:
            r = self.rooms[sorted_keys[0]]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        else:
            self.player_x, self.player_y = 0, 0

    def _create_room(self, x, y):
        rid = y * 100 + x 
        if rid not in self.rooms: self.rooms[rid] = Room(rid, (x, y))
        return rid

    def _connect(self, rid1, rid2, direction):
        rev = {"north":"south", "south":"north", "east":"west", "west":"east"}[direction]
        if rid1 in self.rooms and rid2 in self.rooms:
            self.rooms[rid1].add_door(direction, rid2)
            self.rooms[rid2].add_door(rev, rid1)

    def _disconnect(self, x, y, direction):
        rid = y * 100 + x
        if rid in self.rooms:
            self.rooms[rid].doors.pop(direction, None)
            dx, dy = {'north':(0,-1), 'south':(0,1), 'east':(1,0), 'west':(-1,0)}[direction]
            rev = {'north':'south', 'south':'north', 'east':'west', 'west':'east'}[direction]
            rid2 = (y + dy) * 100 + (x + dx)
            if rid2 in self.rooms:
                self.rooms[rid2].doors.pop(rev, None)

    def _place_fragments(self, placement_dict):
        for (x, y), item in placement_dict.items():
            rid = y * 100 + x
            if rid in self.rooms: self.rooms[rid].fragment = item

    # --- 1. Barbell Map (Optimized V2) ---
    def _setup_barbell(self):
            # 左侧块 (x=0,1, rows=0-3)
            for r in range(4):
                for c in range(2): self._create_room(c, r)
            # 右侧块 (x=4,5, rows=0-3)
            for r in range(4):
                for c in range(4, 6): self._create_room(c, r)
            # 中间走廊 (x=2,3, rows=1,2)
            for r in [1, 2]:
                for c in range(2, 4): self._create_room(c, r)

            # 1. 先建立全连接
            for rid, r in self.rooms.items():
                x, y = r.logical_pos
                if y*100+(x+1) in self.rooms: self._connect(rid, y*100+(x+1), "east")
                if (y+1)*100+x in self.rooms: self._connect(rid, (y+1)*100+x, "south")
                
            # 2. 墙壁设置
            # 注意：移除了之前对四个角落垂直方向的断开设置，使其变回通道
            
            # 走廊设置 (保持之前的要求)
            self._disconnect(2, 1, 'east') # 上方走廊中间断开 (Room 7-8)
            self._disconnect(2, 2, 'west') # Room 13 (2,2) 左侧墙
            self._disconnect(4, 2, 'west') # Room 15 (4,2) 左侧墙
            
            # 3. 放置物品
            self._place_fragments({
                (0,0):'e1', (0,3):'e2', 
                (2,1):'e3', (3,1):'e4', 
                (5,0):'e5', (5,3):'e6' 
            })
            return 0

    # ... (Grid, Path, Ladder 保持不变) ...
    
    # --- 2. Grid Map ---
    def _setup_grid(self):
        for y in range(4):
            for x in range(5):
                rid = self._create_room(x, y)
                if x>0: self._connect(rid, y*100+(x-1), "west")
                if y>0: self._connect(rid, (y-1)*100+x, "north")
        self._place_fragments({(0,0):'e1', (4,0):'e2', (0,3):'e3', (4,3):'e4', (2,0):'e5', (2,3):'e6'})
        return 202

    # --- 3. Path Map ---
    def _setup_path(self):
        path_coords = []
        width = 6
        for x in range(width): path_coords.append((x, 0))
        for x in range(width-1, -1, -1): path_coords.append((x, 1))
        for x in range(width): path_coords.append((x, 2))
        
        prev_rid = None
        for (x, y) in path_coords:
            rid = self._create_room(x, y)
            if prev_rid is not None:
                px, py = self.rooms[prev_rid].logical_pos
                if x>px: self._connect(prev_rid, rid, "east")
                elif x<px: self._connect(prev_rid, rid, "west")
                elif y>py: self._connect(prev_rid, rid, "south")
            prev_rid = rid
            
        items = ['e1', 'e2', 'e3', 'e4', 'e5', 'e6']
        indices = [2, 5, 8, 11, 14, 17]
        for i, idx in enumerate(indices):
            if idx < len(path_coords): 
                r_pos = path_coords[idx]
                self.rooms[r_pos[1]*100+r_pos[0]].fragment = items[i]
        return 0

    # --- 4. Ladder Map (Fixed) ---
    def _setup_ladder(self):
        for x in range(7): self._create_room(x, 0)
        for x in range(7): self._create_room(x, 2)
        self._create_room(3, 1)

        for x in range(6): 
            self._connect(0*100+x, 0*100+(x+1), "east") 
            self._connect(2*100+x, 2*100+(x+1), "east") 
            
        self._connect(0*100+3, 1*100+3, "south") 
        self._connect(1*100+3, 2*100+3, "south")
        
        self._place_fragments({
            (0,0):'e1', (0,2):'e2', 
            (6,0):'e3', (6,2):'e4', 
            (3,0):'e5', (3,2):'e6'
        })
        return 0

    # =========================================================
    # 游戏动作
    # =========================================================
    def check_move_validity(self, cur_x, cur_y, next_x, next_y):
        cur_room = self.get_room_by_grid(cur_x, cur_y)
        next_room = self.get_room_by_grid(next_x, next_y)
        if not next_room: return False, "Void"
        if cur_room.id == next_room.id: return True, "Walk"
        
        lx, ly = cur_room.logical_pos
        nx, ny = next_room.logical_pos
        direction = None
        if nx == lx + 1: direction = "east"
        elif nx == lx - 1: direction = "west"
        elif ny == ly + 1: direction = "south"
        elif ny == ly - 1: direction = "north"
        
        if direction not in cur_room.doors: return False, "Wall"
        
        local_x, local_y = cur_x % 3, cur_y % 3
        is_centered = False
        if direction in ["north", "south"]: is_centered = (local_x == 1)
        elif direction in ["west", "east"]: is_centered = (local_y == 1)
            
        return (True, "Door") if is_centered else (False, "Wall_No_Door")

    def move(self, direction):
        if self.game_over: return
        self.global_counter += 1
        log_dir = self.direction_map.get(direction, direction)
        
        dx, dy = 0, 0
        if direction == "north": dy = -1
        elif direction == "south": dy = 1
        elif direction == "west": dx = -1
        elif direction == "east": dx = 1
        
        target_x, target_y = self.player_x + dx, self.player_y + dy
        valid, reason = self.check_move_validity(self.player_x, self.player_y, target_x, target_y)
        
        if valid:
            self.player_x, self.player_y = target_x, target_y
            new_room = self.get_room_by_grid(target_x, target_y)
            if new_room: new_room.visited = True
            self._log_step(log_dir, reason, Config.REWARD_STEP, True)
        else:
            self.add_log(f"移动失败 ({reason})")
            self._log_step(log_dir, reason, Config.REWARD_INVALID_MOVE, False)

    def pickup(self):
        if self.game_over: return
        self.global_counter += 1
        room = self.get_room_by_grid(self.player_x, self.player_y)
        
        if not room or not room.fragment:
            self.add_log("无物品")
            self._log_step("collect", "None", Config.REWARD_STEP, False)
            return
            
        lx, ly = room.logical_pos
        if (self.player_x, self.player_y) != (lx*3 + 1, ly*3 + 1):
            self.add_log("需走到物品上")
            self._log_step("collect", "Not_On_Item", Config.REWARD_STEP, False)
            return
            
        if len(self.backpack) >= Config.BACKPACK_CAPACITY:
            self.add_log("背包已满")
            self._log_step("collect", "Full", Config.REWARD_STEP, False)
            return
            
        frag = room.fragment
        self.backpack.append(frag)
        self.add_log(f"获得: {frag}")
        self._log_step("collect", frag, Config.REWARD_COLLECT, True)
        self.check_win()
        self._auto_transform()

    def drop_item(self, index):
        self.global_counter += 1
        if 0 <= index < len(self.backpack):
            item = self.backpack.pop(index)
            if index in self.selected_items: self.selected_items.remove(index)
            self.selected_items = [i if i < index else i-1 for i in self.selected_items]
            self.add_log(f"丢弃: {item}")
            self._log_step("ditch", item, Config.REWARD_DITCH, True)
            return True
        self._log_step("ditch", "Invalid_Index", Config.REWARD_STEP, False)
        return False

    def toggle_select(self, index):
        if index >= len(self.backpack): return
        if index in self.selected_items: self.selected_items.remove(index)
        else:
            if len(self.selected_items) < 2: self.selected_items.append(index)
            else: self.selected_items.pop(0); self.selected_items.append(index)

    def attempt_synthesis(self):
        self.global_counter += 1
        if len(self.selected_items) == 1:
            self.add_log("单物品将自动转化 (如有)")
            self.selected_items = []
            return

        if len(self.selected_items) == 2:
            idx1, idx2 = self.selected_items
            if idx1 >= len(self.backpack) or idx2 >= len(self.backpack):
                self.selected_items = []; return

            item1, item2 = self.backpack[idx1], self.backpack[idx2]
            result = None
            for res, ingredients in RECIPES_AND.items():
                if sorted([item1, item2]) == sorted(ingredients):
                    result = res
                    break
            
            if result:
                for idx in sorted([idx1, idx2], reverse=True): self.backpack.pop(idx)
                self.backpack.append(result)
                self.selected_items = []
                self.add_log(f"合成: {result}")
                self._log_step("merge", f"{item1}+{item2}->{result}", Config.REWARD_MERGE, True)
                self.check_win()
                self._auto_transform()
            else:
                self.add_log("合成失败")
                self._log_step("merge", f"{item1}+{item2}_Fail", Config.REWARD_STEP, False)
                self.selected_items = []
            return
        
        self.add_log("请选择 2 个物品进行合成")

    def check_win(self):
        if 'A' in self.backpack and not self.game_over:
            self.game_over = True
            self.win_reason = "任务完成 [A]！"
            if len(self.recorder.memory_buffer) > 0:
                self.recorder.memory_buffer[-1]["Reward"] = Config.REWARD_WIN
                self.recorder.memory_buffer[-1]["Game_Over"] = True
            self.add_log("!!! 胜利 !!!")

    def reset(self, new_map_type=None):
        m_type = new_map_type if new_map_type else self.map_type
        self.recorder.save_to_file()
        self.__init__(self.recorder, m_type)

# ==========================================
# 渲染系统
# ==========================================
def draw_ui(game, screen, mouse_pos):
    screen.fill(Config.COLOR_BG)
    
    panel_left = pygame.Rect(20, 20, 280, Config.SCREEN_HEIGHT - 40)
    panel_right = pygame.Rect(Config.SCREEN_WIDTH - 420, 20, 400, Config.SCREEN_HEIGHT - 40)
    map_area = pygame.Rect(320, 20, Config.SCREEN_WIDTH - 760, Config.SCREEN_HEIGHT - 40)
    
    draw_panel(screen, panel_left, "Auto-Seq 控制台")
    y_off = 70
    
    # 状态信息
    screen.blit(font_lg.render(f"Map: {game.map_type}", True, Config.COLOR_HIGHLIGHT), (panel_left.x+20, y_off))
    y_off += 35
    cur_room = game.get_room_by_grid(game.player_x, game.player_y)
    # [修改] 显示连续ID
    rid_str = str(cur_room.seq_id) if cur_room else "Void"
    screen.blit(font_md.render(f"当前房间: {rid_str}", True, Config.COLOR_ACCENT), (panel_left.x+20, y_off))
    y_off += 30
    screen.blit(font_sm.render(f"Grid Pos: ({game.player_x}, {game.player_y})", True, Config.COLOR_TEXT_DIM), (panel_left.x+20, y_off))
    y_off += 25
    
    info_lines = [f"步数: {game.global_counter}", "按键: WASD移动", "F:拾取, 右键:选择", "1-4: 切换地图"]
    for line in info_lines:
        screen.blit(font_sm.render(line, True, Config.COLOR_TEXT_MAIN), (panel_left.x+20, y_off))
        y_off += 22
        
    y_off += 20
    screen.blit(font_md.render("自动化公式:", True, Config.COLOR_TEXT_DIM), (panel_left.x + 20, y_off))
    y_off += 25
    
    col_and = (150, 200, 150)
    col_auto = (255, 200, 100)
    
    def draw_formula_group(lines, color, y_start, tag=""):
        for f in lines:
            ft = font_sm.render(f + tag, True, color)
            screen.blit(ft, (panel_left.x + 20, y_start))
            y_start += 18 
        return y_start + 8

    y_off = draw_formula_group(["e1+e2→b1", "e2+e3→b2", "e1+e3→c1"], col_and, y_off)
    y_off = draw_formula_group(["e4+e6→c2", "e4+e5→d1", "e5+e6→d2"], col_and, y_off)
    y_off = draw_formula_group(["b1/b2 → [Auto] B"], col_auto, y_off)
    y_off = draw_formula_group(["c1/c2 → [Auto] C"], col_auto, y_off)
    y_off = draw_formula_group(["d1/d2 → [Auto] D"], col_auto, y_off)
    y_off = draw_formula_group(["B+C → a1", "C+D → a2"], col_and, y_off)
    y_off = draw_formula_group(["a1/a2 → [Auto] A"], col_auto, y_off)

    # --- 地图绘制 ---
    all_lx = [r.logical_pos[0] for r in game.rooms.values()]
    all_ly = [r.logical_pos[1] for r in game.rooms.values()]
    
    if all_lx and all_ly:
        min_lx, max_lx = min(all_lx), max(all_lx)
        min_ly, max_ly = min(all_ly), max(all_ly)
        grid_w_cells = (max_lx - min_lx + 1) * 3
        grid_h_cells = (max_ly - min_ly + 1) * 3
        grid_px_w = grid_w_cells * Config.GRID_SIZE
        grid_px_h = grid_h_cells * Config.GRID_SIZE
        center_off_x = map_area.centerx - grid_px_w // 2 - (min_lx * 3) * Config.GRID_SIZE
        center_off_y = map_area.centery - grid_px_h // 2 - (min_ly * 3) * Config.GRID_SIZE
    else: center_off_x, center_off_y = 0, 0

    game.door_rects = {}
    
    # 1. 地面
    for rid, room in game.rooms.items():
        lx, ly = room.logical_pos
        for dy in range(3):
            for dx in range(3):
                gx, gy = lx * 3 + dx, ly * 3 + dy
                sx = center_off_x + gx * Config.GRID_SIZE
                sy = center_off_y + gy * Config.GRID_SIZE
                cell_rect = pygame.Rect(sx, sy, Config.GRID_SIZE, Config.GRID_SIZE)
                
                bg_col = Config.COLOR_ROOM_BG
                if room.visited: bg_col = Config.COLOR_ROOM_VISITED
                # 使用 seq_id 判断当前房间
                if room.seq_id == game.get_current_room_seq_id(): bg_col = (85, 95, 115)
                
                pygame.draw.rect(screen, bg_col, cell_rect)
                pygame.draw.rect(screen, (60, 60, 70), cell_rect, 1)

    # 2. 墙壁
    wall_thickness = 4
    wall_color = Config.COLOR_ROOM_WALL
    
    for rid, room in game.rooms.items():
        lx, ly = room.logical_pos
        rx = center_off_x + lx * 3 * Config.GRID_SIZE
        ry = center_off_y + ly * 3 * Config.GRID_SIZE
        size = 3 * Config.GRID_SIZE 
        
        tl = (rx, ry)
        tr = (rx + size, ry)
        bl = (rx, ry + size)
        br = (rx + size, ry + size)
        
        s = Config.GRID_SIZE
        
        if 'north' in room.doors:
            pygame.draw.line(screen, wall_color, tl, (rx + s, ry), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + 2*s, ry), tr, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tl, tr, wall_thickness)
            
        if 'south' in room.doors:
            pygame.draw.line(screen, wall_color, bl, (rx + s, ry + size), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + 2*s, ry + size), br, wall_thickness)
        else: pygame.draw.line(screen, wall_color, bl, br, wall_thickness)
            
        if 'west' in room.doors:
            pygame.draw.line(screen, wall_color, tl, (rx, ry + s), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx, ry + 2*s), bl, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tl, bl, wall_thickness)
            
        if 'east' in room.doors:
            pygame.draw.line(screen, wall_color, tr, (rx + size, ry + s), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + size, ry + 2*s), br, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tr, br, wall_thickness)

        # [修改] 绘制房间ID编号，方便查看顺序
        # 在房间左上角画一个小数字
        id_surf = font_sm.render(str(room.seq_id), True, (100, 110, 130))
        screen.blit(id_surf, (rx + 5, ry + 5))

        if room.fragment:
            cx = rx + 1.5 * s
            cy = ry + 1.5 * s
            f_col = Config.FRAGMENT_COLORS.get(room.fragment, (200,200,200))
            pygame.draw.circle(screen, f_col, (cx, cy), s//2 - 4)
            ft = font_sm.render(room.fragment, True, (0,0,0))
            screen.blit(ft, (cx-ft.get_width()//2, cy-ft.get_height()//2))

    # 3. 玩家
    p_sx = center_off_x + game.player_x * Config.GRID_SIZE
    p_sy = center_off_y + game.player_y * Config.GRID_SIZE
    pygame.draw.rect(screen, Config.COLOR_PLAYER, (p_sx+4, p_sy+4, Config.GRID_SIZE-8, Config.GRID_SIZE-8), border_radius=3)

    # --- 右侧面板 ---
    draw_panel(screen, panel_right, "背包 (2格合成)")
    bp_start_y = panel_right.y + 60
    game.backpack_rects = {}
    game.drop_rects = {}
    
    for i in range(Config.BACKPACK_CAPACITY):
        col, row = i % 2, i // 2
        bx, by = panel_right.x + 50 + col * 100, bp_start_y + row * 100
        rect = pygame.Rect(bx, by, 80, 80)
        game.backpack_rects[i] = rect
        
        color = (50, 56, 70)
        if i in game.selected_items: color = Config.COLOR_HIGHLIGHT
        pygame.draw.rect(screen, color, rect, border_radius=8)
        pygame.draw.rect(screen, Config.COLOR_BORDER, rect, 2, border_radius=8)
        
        if i < len(game.backpack):
            item = game.backpack[i]
            ic = Config.FRAGMENT_COLORS.get(item, (200,200,200))
            pygame.draw.rect(screen, ic, (bx+15, by+15, 50, 30), border_radius=4)
            ts = font_sm.render(item, True, (0,0,0))
            screen.blit(ts, (bx+40-ts.get_width()//2, by+20))
            
            dr = pygame.Rect(bx+45, by+50, 30, 20)
            game.drop_rects[i] = dr
            pygame.draw.rect(screen, (150, 80, 80), dr, border_radius=3)
            dt = font_sm.render("丢", True, (255,255,255))
            screen.blit(dt, (dr.centerx-dt.get_width()//2, dr.centery-dt.get_height()//2))

    btn_rect = pygame.Rect(panel_right.x + 30, bp_start_y + 220, 340, 50)
    can_syn = (len(game.selected_items) == 2)
    pygame.draw.rect(screen, Config.COLOR_SUCCESS if can_syn else (70,70,80), btn_rect, border_radius=8)
    btn_txt = font_md.render("合成选中 (And)", True, (255,255,255))
    screen.blit(btn_txt, (btn_rect.centerx-btn_txt.get_width()//2, btn_rect.centery-btn_txt.get_height()//2))
    game.synth_button_rect = btn_rect
    
    ly = bp_start_y + 300
    for log in list(game.logs)[::-1]:
        screen.blit(font_sm.render(log, True, Config.COLOR_TEXT_DIM), (panel_right.x+20, ly))
        ly += 20

    if game.game_over:
        ov = pygame.Surface((Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT), pygame.SRCALPHA)
        ov.fill((0,0,0,180))
        screen.blit(ov, (0,0))
        txt = font_xl.render(game.win_reason, True, Config.COLOR_SUCCESS)
        screen.blit(txt, (Config.SCREEN_WIDTH//2 - txt.get_width()//2, Config.SCREEN_HEIGHT//2))

def draw_panel(screen, rect, title):
    pygame.draw.rect(screen, Config.COLOR_PANEL, rect, border_radius=15)
    pygame.draw.rect(screen, Config.COLOR_BORDER, rect, 2, border_radius=15)
    screen.blit(font_lg.render(title, True, Config.COLOR_TEXT_MAIN), (rect.x + 20, rect.y + 15))

def main():
    recorder = RLDataRecorder("RL_Auto_Seq_User")
    game = Game(recorder, map_type="Barbell")
    clock = pygame.time.Clock()
    running = True
    pygame.key.set_repeat(200, 50) 
    
    try:
        while running:
            mouse_pos = pygame.mouse.get_pos()
            for event in pygame.event.get():
                if event.type == pygame.QUIT: running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_1: game.reset("Barbell")
                    elif event.key == pygame.K_2: game.reset("Grid")
                    elif event.key == pygame.K_3: game.reset("Path")
                    elif event.key == pygame.K_4: game.reset("Ladder")
                    elif not game.game_over:
                        if event.key == pygame.K_f: game.pickup()
                        elif event.key == pygame.K_r: game.reset()
                        elif event.key in [pygame.K_w, pygame.K_UP]: game.move("north")
                        elif event.key in [pygame.K_s, pygame.K_DOWN]: game.move("south")
                        elif event.key in [pygame.K_a, pygame.K_LEFT]: game.move("west")
                        elif event.key in [pygame.K_d, pygame.K_RIGHT]: game.move("east")
                    elif game.game_over: game.reset()

                if event.type == pygame.MOUSEBUTTONDOWN:
                    if game.game_over: game.reset(); continue
                    handled = False
                    for i, r in game.drop_rects.items():
                        if r.collidepoint(event.pos): game.drop_item(i); handled = True; break
                    if not handled:
                        for i, r in game.backpack_rects.items():
                            if r.collidepoint(event.pos): game.toggle_select(i); handled = True; break
                    if not handled and game.synth_button_rect.collidepoint(event.pos):
                        game.attempt_synthesis(); handled = True

            draw_ui(game, screen, mouse_pos)
            pygame.display.flip()
            clock.tick(60)

    except KeyboardInterrupt: pass
    finally:
        game.recorder.save_to_file()
        pygame.quit()
        sys.exit()

if __name__ == "__main__":
    main()

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


pygame 2.5.2 (SDL 2.32.56, Python 3.11.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
Data Recorder Initialized. Output directory: rl_data/20251214_104959
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251214_104959/game_log_RL_Auto_Seq_User.csv
Excel error: No module na

SystemExit: 

d:\ANACONDA\envs\yh311_G\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import pygame
import sys
import random
import datetime
import os
import csv
from collections import deque

# 尝试导入 pandas
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# ==========================================
# 配置与常量定义
# ==========================================
class Config:
    SCREEN_WIDTH = 1600
    SCREEN_HEIGHT = 900
    TITLE = "Project: Minecraft RL (Auto-Logic + Seq IDs + Assets)"
    
    GRID_SIZE = 30  
    ROOM_GRID_WIDTH = 3
    ROOM_GRID_HEIGHT = 3
    
    BACKPACK_CAPACITY = 4
    
    # 你的素材路径 (使用了 raw string r"" 以防止转义字符问题)
    # 如果你在不同电脑运行，请确保此路径存在，或者改为相对路径
    ASSET_PATH = r"D:\桌面文件夹\华东师范\Minecraft8.0\assets"
    
    # RL Rewards
    REWARD_STEP = -0.1          
    REWARD_INVALID_MOVE = -0.5  
    REWARD_COLLECT = 0.5        
    REWARD_DITCH = -0.2         
    REWARD_TRANSFORM = 1.0      
    REWARD_MERGE = 2.0          
    REWARD_WIN = 10.0           
    
    # 颜色定义
    COLOR_BG = (46, 52, 64)
    COLOR_PANEL = (59, 66, 82)
    COLOR_BORDER = (76, 86, 106)
    COLOR_TEXT_MAIN = (236, 239, 244)
    COLOR_TEXT_DIM = (216, 222, 233)
    COLOR_ACCENT = (136, 192, 208)
    COLOR_SUCCESS = (163, 190, 140)
    COLOR_HIGHLIGHT = (94, 129, 172)
    COLOR_PLAYER = (255, 80, 80)
    
    # 房间配色
    COLOR_ROOM_BG = (67, 76, 94)      
    COLOR_ROOM_WALL = (200, 210, 220) 
    COLOR_ROOM_VISITED = (70, 80, 100)
    
    # 兜底颜色（当图片加载失败或不存在时使用）
    FRAGMENT_COLORS = {
        'e1': (255, 100, 100), 'e2': (100, 200, 100), 'e3': (100, 100, 255),
        'e4': (255, 255, 100), 'e5': (255, 100, 255), 'e6': (100, 255, 255),
        'b1': (205, 170, 125), 'b2': (150, 200, 150), 
        'c1': (150, 150, 200), 'c2': (220, 220, 150),
        'd1': (200, 150, 200), 'd2': (150, 200, 200),
        'B': (140, 140, 140), 'C': (180, 180, 180), 'D': (220, 220, 220),
        'a1': (255, 160, 80), 'a2': (60, 160, 255),
        'A': (255, 215, 0)
    }

# 初始化 Pygame
pygame.init()
screen = pygame.display.set_mode((Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT))
pygame.display.set_caption(Config.TITLE)

def get_font(size):
    try: return pygame.font.SysFont('simhei', size)
    except: return pygame.font.SysFont('arial', size)

font_sm = get_font(16)
font_md = get_font(20)
font_lg = get_font(28)
font_xl = get_font(40)

# ==========================================
# 资源管理器 (新增)
# ==========================================
class AssetManager:
    def __init__(self):
        self.images = {}
        # 需要加载图片的项目列表
        self.item_list = [
            'e1', 'e2', 'e3', 'e4', 'e5', 'e6',
            'b1', 'b2', 'c1', 'c2', 'd1', 'd2',
            'a1', 'a2', 'A', 'B', 'C', 'D' # 尝试加载这些，虽然你可能没有所有图片
        ]
        self.load_assets()

    def load_assets(self):
        if not os.path.exists(Config.ASSET_PATH):
            print(f"[警告] 资源路径不存在: {Config.ASSET_PATH}")
            return

        print(f"正在从 {Config.ASSET_PATH} 加载资源...")
        for item in self.item_list:
            # 尝试加载 png
            file_path = os.path.join(Config.ASSET_PATH, f"{item}.png")
            if os.path.exists(file_path):
                try:
                    img = pygame.image.load(file_path).convert_alpha()
                    self.images[item] = img
                    print(f"  -> 已加载: {item}.png")
                except Exception as e:
                    print(f"  [错误] 加载 {item}.png 失败: {e}")
            else:
                pass
                # print(f"  [跳过] 未找到 {item}.png")

    def get_image(self, item_name, width, height):
        """获取并缩放图片，如果不存在则返回 None"""
        if item_name in self.images:
            return pygame.transform.smoothscale(self.images[item_name], (width, height))
        return None

# 初始化全局资源管理器
asset_manager = AssetManager()

# ==========================================
# 核心逻辑 (保持不变)
# ==========================================

RECIPES_AND = {
    'b1': ['e1', 'e2'], 'b2': ['e2', 'e3'],
    'c1': ['e1', 'e3'], 'c2': ['e4', 'e6'],
    'd1': ['e4', 'e5'], 'd2': ['e5', 'e6'],
    'a1': ['B', 'C'],   'a2': ['C', 'D'],
}

RECIPES_OR = {
    'b1': 'B', 'b2': 'B',
    'c1': 'C', 'c2': 'C',
    'd1': 'D', 'd2': 'D',
    'a1': 'A', 'a2': 'A'
}

class RLDataRecorder:
    def __init__(self, participant_id="RL_Agent_01"):
        self.participant_id = participant_id
        self.timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.data_dir = f"rl_data/{self.timestamp_str}"
        os.makedirs(self.data_dir, exist_ok=True)
        self.memory_buffer = [] 
        self.episode_count = 0
        self.step_count = 0
        print(f"Data Recorder Initialized. Output directory: {self.data_dir}")

    def start_episode(self):
        self.episode_count += 1
        self.step_count = 0
    
    def log_action(self, episode, step, map_type, current_room_id, pos_x, pos_y, 
                   backpack, action_name, action_detail, reward, is_valid, game_over):
        self.step_count += 1
        record = {
            "Participant": self.participant_id,
            "Episode_ID": episode,
            "Step_Index": step,
            "Timestamp": datetime.datetime.now().isoformat(),
            "Map_Structure": map_type,
            "Current_Room_ID": current_room_id, 
            "Grid_X": pos_x,
            "Grid_Y": pos_y,
            "Backpack_Content": str(backpack),
            "Backpack_Count": len(backpack),
            "Action_Type": action_name,
            "Action_Detail": str(action_detail),
            "Action_Valid": is_valid,
            "Reward": reward,
            "Game_Over": game_over
        }
        self.memory_buffer.append(record)

    def save_to_file(self):
        if not self.memory_buffer: return
        filename_base = f"{self.data_dir}/game_log_{self.participant_id}"
        if HAS_PANDAS:
            try:
                pd.DataFrame(self.memory_buffer).to_excel(f"{filename_base}.xlsx", index=False)
                print(f"Data saved to Excel: {filename_base}.xlsx")
                return
            except Exception as e: print(f"Excel error: {e}")
        
        try:
            keys = self.memory_buffer[0].keys()
            with open(f"{filename_base}.csv", 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=keys)
                writer.writeheader()
                writer.writerows(self.memory_buffer)
            print(f"Data saved to CSV: {filename_base}.csv")
        except Exception as e: print(f"CSV error: {e}")

class Room:
    def __init__(self, id, logical_pos):
        self.id = id           # 原始坐标ID (e.g., 201)
        self.seq_id = 0        # 顺序ID (e.g., 1, 2, 3...)
        self.logical_pos = logical_pos 
        self.fragment = None 
        self.doors = {}
        self.visited = False
    
    def add_door(self, direction, target_id):
        self.doors[direction] = target_id

class Game:
    def __init__(self, recorder, map_type="Barbell"):
        self.recorder = recorder
        self.map_type = map_type
        self.rooms = {}
        self.player_x = 0
        self.player_y = 0
        self.backpack = []
        self.global_counter = 0
        self.game_over = False
        self.win_reason = ""
        self.logs = deque(maxlen=8)
        self.selected_items = []
        
        self.direction_map = {"north": "Up", "south": "Down", "west": "Left", "east": "Right"}
        self.setup_level(map_type)
        self.recorder.start_episode()
        self.add_log(f"Map: {map_type} (Seq IDs)")

    def add_log(self, text):
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        self.logs.append(f"[{timestamp}] {text}")

    def get_room_by_grid(self, gx, gy):
        lx, ly = gx // 3, gy // 3
        rid = ly * 100 + lx
        return self.rooms.get(rid)

    def get_current_room_seq_id(self):
        r = self.get_room_by_grid(self.player_x, self.player_y)
        return r.seq_id if r else -1

    def _log_step(self, action_name, action_detail, reward, is_valid):
        self.recorder.log_action(
            self.recorder.episode_count, self.global_counter, self.map_type,
            self.get_current_room_seq_id(), self.player_x, self.player_y,
            self.backpack.copy(), action_name, action_detail, reward, is_valid, self.game_over
        )

    def _auto_transform(self):
        changed = False
        for i in range(len(self.backpack)):
            item = self.backpack[i]
            if item in RECIPES_OR:
                new_item = RECIPES_OR[item]
                self.backpack[i] = new_item
                self.add_log(f"自动转化: {item} -> {new_item}")
                self.global_counter += 1 
                self._log_step("auto_merge", f"{item}->{new_item}", Config.REWARD_TRANSFORM, True)
                changed = True
        
        if changed:
            self.check_win()
            self._auto_transform()

    # =========================================================
    # 地图生成
    # =========================================================
    def setup_level(self, map_type):
        self.rooms = {}
        self.map_type = map_type
        start_rid = 0
        
        if map_type == "Barbell": start_rid = self._setup_barbell()
        elif map_type == "Grid": start_rid = self._setup_grid()
        elif map_type == "Path": start_rid = self._setup_path()
        elif map_type == "Ladder": start_rid = self._setup_ladder()
        
        sorted_keys = sorted(self.rooms.keys())
        for idx, key in enumerate(sorted_keys):
            self.rooms[key].seq_id = idx + 1

        if start_rid in self.rooms:
            r = self.rooms[start_rid]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        elif self.rooms:
            r = self.rooms[sorted_keys[0]]
            self.player_x, self.player_y = r.logical_pos[0]*3 + 1, r.logical_pos[1]*3 + 1
            r.visited = True
        else:
            self.player_x, self.player_y = 0, 0

    def _create_room(self, x, y):
        rid = y * 100 + x 
        if rid not in self.rooms: self.rooms[rid] = Room(rid, (x, y))
        return rid

    def _connect(self, rid1, rid2, direction):
        rev = {"north":"south", "south":"north", "east":"west", "west":"east"}[direction]
        if rid1 in self.rooms and rid2 in self.rooms:
            self.rooms[rid1].add_door(direction, rid2)
            self.rooms[rid2].add_door(rev, rid1)

    def _disconnect(self, x, y, direction):
        rid = y * 100 + x
        if rid in self.rooms:
            self.rooms[rid].doors.pop(direction, None)
            dx, dy = {'north':(0,-1), 'south':(0,1), 'east':(1,0), 'west':(-1,0)}[direction]
            rev = {'north':'south', 'south':'north', 'east':'west', 'west':'east'}[direction]
            rid2 = (y + dy) * 100 + (x + dx)
            if rid2 in self.rooms:
                self.rooms[rid2].doors.pop(rev, None)

    def _place_fragments(self, placement_dict):
        for (x, y), item in placement_dict.items():
            rid = y * 100 + x
            if rid in self.rooms: self.rooms[rid].fragment = item

    # --- 1. Barbell Map ---
    def _setup_barbell(self):
            for r in range(4):
                for c in range(2): self._create_room(c, r)
            for r in range(4):
                for c in range(4, 6): self._create_room(c, r)
            for r in [1, 2]:
                for c in range(2, 4): self._create_room(c, r)

            for rid, r in self.rooms.items():
                x, y = r.logical_pos
                if y*100+(x+1) in self.rooms: self._connect(rid, y*100+(x+1), "east")
                if (y+1)*100+x in self.rooms: self._connect(rid, (y+1)*100+x, "south")
            
            self._disconnect(2, 1, 'east') 
            self._disconnect(2, 2, 'west') 
            self._disconnect(4, 2, 'west') 
            
            self._place_fragments({
                (0,0):'e1', (0,3):'e2', 
                (2,1):'e3', (3,1):'e4', 
                (5,0):'e5', (5,3):'e6' 
            })
            return 0

    # --- 2. Grid Map ---
    def _setup_grid(self):
        for y in range(4):
            for x in range(5):
                rid = self._create_room(x, y)
                if x>0: self._connect(rid, y*100+(x-1), "west")
                if y>0: self._connect(rid, (y-1)*100+x, "north")
        self._place_fragments({(0,0):'e1', (4,0):'e2', (0,3):'e3', (4,3):'e4', (2,0):'e5', (2,3):'e6'})
        return 202

    # --- 3. Path Map ---
    def _setup_path(self):
        path_coords = []
        width = 6
        for x in range(width): path_coords.append((x, 0))
        for x in range(width-1, -1, -1): path_coords.append((x, 1))
        for x in range(width): path_coords.append((x, 2))
        
        prev_rid = None
        for (x, y) in path_coords:
            rid = self._create_room(x, y)
            if prev_rid is not None:
                px, py = self.rooms[prev_rid].logical_pos
                if x>px: self._connect(prev_rid, rid, "east")
                elif x<px: self._connect(prev_rid, rid, "west")
                elif y>py: self._connect(prev_rid, rid, "south")
            prev_rid = rid
            
        items = ['e1', 'e2', 'e3', 'e4', 'e5', 'e6']
        indices = [2, 5, 8, 11, 14, 17]
        for i, idx in enumerate(indices):
            if idx < len(path_coords): 
                r_pos = path_coords[idx]
                self.rooms[r_pos[1]*100+r_pos[0]].fragment = items[i]
        return 0

    # --- 4. Ladder Map ---
    def _setup_ladder(self):
        for x in range(7): self._create_room(x, 0)
        for x in range(7): self._create_room(x, 2)
        self._create_room(3, 1)

        for x in range(6): 
            self._connect(0*100+x, 0*100+(x+1), "east") 
            self._connect(2*100+x, 2*100+(x+1), "east") 
            
        self._connect(0*100+3, 1*100+3, "south") 
        self._connect(1*100+3, 2*100+3, "south")
        
        self._place_fragments({
            (0,0):'e1', (0,2):'e2', 
            (6,0):'e3', (6,2):'e4', 
            (3,0):'e5', (3,2):'e6'
        })
        return 0

    # =========================================================
    # 游戏动作
    # =========================================================
    def check_move_validity(self, cur_x, cur_y, next_x, next_y):
        cur_room = self.get_room_by_grid(cur_x, cur_y)
        next_room = self.get_room_by_grid(next_x, next_y)
        if not next_room: return False, "Void"
        if cur_room.id == next_room.id: return True, "Walk"
        
        lx, ly = cur_room.logical_pos
        nx, ny = next_room.logical_pos
        direction = None
        if nx == lx + 1: direction = "east"
        elif nx == lx - 1: direction = "west"
        elif ny == ly + 1: direction = "south"
        elif ny == ly - 1: direction = "north"
        
        if direction not in cur_room.doors: return False, "Wall"
        
        local_x, local_y = cur_x % 3, cur_y % 3
        is_centered = False
        if direction in ["north", "south"]: is_centered = (local_x == 1)
        elif direction in ["west", "east"]: is_centered = (local_y == 1)
            
        return (True, "Door") if is_centered else (False, "Wall_No_Door")

    def move(self, direction):
        if self.game_over: return
        self.global_counter += 1
        log_dir = self.direction_map.get(direction, direction)
        
        dx, dy = 0, 0
        if direction == "north": dy = -1
        elif direction == "south": dy = 1
        elif direction == "west": dx = -1
        elif direction == "east": dx = 1
        
        target_x, target_y = self.player_x + dx, self.player_y + dy
        valid, reason = self.check_move_validity(self.player_x, self.player_y, target_x, target_y)
        
        if valid:
            self.player_x, self.player_y = target_x, target_y
            new_room = self.get_room_by_grid(target_x, target_y)
            if new_room: new_room.visited = True
            self._log_step(log_dir, reason, Config.REWARD_STEP, True)
        else:
            self.add_log(f"移动失败 ({reason})")
            self._log_step(log_dir, reason, Config.REWARD_INVALID_MOVE, False)

    def pickup(self):
        if self.game_over: return
        self.global_counter += 1
        room = self.get_room_by_grid(self.player_x, self.player_y)
        
        if not room or not room.fragment:
            self.add_log("无物品")
            self._log_step("collect", "None", Config.REWARD_STEP, False)
            return
            
        lx, ly = room.logical_pos
        if (self.player_x, self.player_y) != (lx*3 + 1, ly*3 + 1):
            self.add_log("需走到物品上")
            self._log_step("collect", "Not_On_Item", Config.REWARD_STEP, False)
            return
            
        if len(self.backpack) >= Config.BACKPACK_CAPACITY:
            self.add_log("背包已满")
            self._log_step("collect", "Full", Config.REWARD_STEP, False)
            return
            
        frag = room.fragment
        self.backpack.append(frag)
        self.add_log(f"获得: {frag}")
        self._log_step("collect", frag, Config.REWARD_COLLECT, True)
        self.check_win()
        self._auto_transform()

    def drop_item(self, index):
        self.global_counter += 1
        if 0 <= index < len(self.backpack):
            item = self.backpack.pop(index)
            if index in self.selected_items: self.selected_items.remove(index)
            self.selected_items = [i if i < index else i-1 for i in self.selected_items]
            self.add_log(f"丢弃: {item}")
            self._log_step("ditch", item, Config.REWARD_DITCH, True)
            return True
        self._log_step("ditch", "Invalid_Index", Config.REWARD_STEP, False)
        return False

    def toggle_select(self, index):
        if index >= len(self.backpack): return
        if index in self.selected_items: self.selected_items.remove(index)
        else:
            if len(self.selected_items) < 2: self.selected_items.append(index)
            else: self.selected_items.pop(0); self.selected_items.append(index)

    def attempt_synthesis(self):
        self.global_counter += 1
        if len(self.selected_items) == 1:
            self.add_log("单物品将自动转化 (如有)")
            self.selected_items = []
            return

        if len(self.selected_items) == 2:
            idx1, idx2 = self.selected_items
            if idx1 >= len(self.backpack) or idx2 >= len(self.backpack):
                self.selected_items = []; return

            item1, item2 = self.backpack[idx1], self.backpack[idx2]
            result = None
            for res, ingredients in RECIPES_AND.items():
                if sorted([item1, item2]) == sorted(ingredients):
                    result = res
                    break
            
            if result:
                for idx in sorted([idx1, idx2], reverse=True): self.backpack.pop(idx)
                self.backpack.append(result)
                self.selected_items = []
                self.add_log(f"合成: {result}")
                self._log_step("merge", f"{item1}+{item2}->{result}", Config.REWARD_MERGE, True)
                self.check_win()
                self._auto_transform()
            else:
                self.add_log("合成失败")
                self._log_step("merge", f"{item1}+{item2}_Fail", Config.REWARD_STEP, False)
                self.selected_items = []
            return
        
        self.add_log("请选择 2 个物品进行合成")

    def check_win(self):
        if 'A' in self.backpack and not self.game_over:
            self.game_over = True
            self.win_reason = "任务完成 [A]！"
            if len(self.recorder.memory_buffer) > 0:
                self.recorder.memory_buffer[-1]["Reward"] = Config.REWARD_WIN
                self.recorder.memory_buffer[-1]["Game_Over"] = True
            self.add_log("!!! 胜利 !!!")

    def reset(self, new_map_type=None):
        m_type = new_map_type if new_map_type else self.map_type
        self.recorder.save_to_file()
        self.__init__(self.recorder, m_type)

# ==========================================
# 渲染系统
# ==========================================
def draw_ui(game, screen, mouse_pos):
    screen.fill(Config.COLOR_BG)
    
    panel_left = pygame.Rect(20, 20, 280, Config.SCREEN_HEIGHT - 40)
    panel_right = pygame.Rect(Config.SCREEN_WIDTH - 420, 20, 400, Config.SCREEN_HEIGHT - 40)
    map_area = pygame.Rect(320, 20, Config.SCREEN_WIDTH - 760, Config.SCREEN_HEIGHT - 40)
    
    draw_panel(screen, panel_left, "Auto-Seq 控制台")
    y_off = 70
    
    # 状态信息
    screen.blit(font_lg.render(f"Map: {game.map_type}", True, Config.COLOR_HIGHLIGHT), (panel_left.x+20, y_off))
    y_off += 35
    cur_room = game.get_room_by_grid(game.player_x, game.player_y)
    rid_str = str(cur_room.seq_id) if cur_room else "Void"
    screen.blit(font_md.render(f"当前房间: {rid_str}", True, Config.COLOR_ACCENT), (panel_left.x+20, y_off))
    y_off += 30
    screen.blit(font_sm.render(f"Grid Pos: ({game.player_x}, {game.player_y})", True, Config.COLOR_TEXT_DIM), (panel_left.x+20, y_off))
    y_off += 25
    
    info_lines = [f"步数: {game.global_counter}", "按键: WASD移动", "F:拾取, 右键:选择", "1-4: 切换地图"]
    for line in info_lines:
        screen.blit(font_sm.render(line, True, Config.COLOR_TEXT_MAIN), (panel_left.x+20, y_off))
        y_off += 22
        
    y_off += 20
    screen.blit(font_md.render("自动化公式:", True, Config.COLOR_TEXT_DIM), (panel_left.x + 20, y_off))
    y_off += 25
    
    col_and = (150, 200, 150)
    col_auto = (255, 200, 100)
    
    def draw_formula_group(lines, color, y_start, tag=""):
        for f in lines:
            ft = font_sm.render(f + tag, True, color)
            screen.blit(ft, (panel_left.x + 20, y_start))
            y_start += 18 
        return y_start + 8

    y_off = draw_formula_group(["e1+e2→b1", "e2+e3→b2", "e1+e3→c1"], col_and, y_off)
    y_off = draw_formula_group(["e4+e6→c2", "e4+e5→d1", "e5+e6→d2"], col_and, y_off)
    y_off = draw_formula_group(["b1/b2 → [Auto] B"], col_auto, y_off)
    y_off = draw_formula_group(["c1/c2 → [Auto] C"], col_auto, y_off)
    y_off = draw_formula_group(["d1/d2 → [Auto] D"], col_auto, y_off)
    y_off = draw_formula_group(["B+C → a1", "C+D → a2"], col_and, y_off)
    y_off = draw_formula_group(["a1/a2 → [Auto] A"], col_auto, y_off)

    # --- 地图绘制 ---
    all_lx = [r.logical_pos[0] for r in game.rooms.values()]
    all_ly = [r.logical_pos[1] for r in game.rooms.values()]
    
    if all_lx and all_ly:
        min_lx, max_lx = min(all_lx), max(all_lx)
        min_ly, max_ly = min(all_ly), max(all_ly)
        grid_w_cells = (max_lx - min_lx + 1) * 3
        grid_h_cells = (max_ly - min_ly + 1) * 3
        grid_px_w = grid_w_cells * Config.GRID_SIZE
        grid_px_h = grid_h_cells * Config.GRID_SIZE
        center_off_x = map_area.centerx - grid_px_w // 2 - (min_lx * 3) * Config.GRID_SIZE
        center_off_y = map_area.centery - grid_px_h // 2 - (min_ly * 3) * Config.GRID_SIZE
    else: center_off_x, center_off_y = 0, 0

    game.door_rects = {}
    
    # 1. 地面
    for rid, room in game.rooms.items():
        lx, ly = room.logical_pos
        for dy in range(3):
            for dx in range(3):
                gx, gy = lx * 3 + dx, ly * 3 + dy
                sx = center_off_x + gx * Config.GRID_SIZE
                sy = center_off_y + gy * Config.GRID_SIZE
                cell_rect = pygame.Rect(sx, sy, Config.GRID_SIZE, Config.GRID_SIZE)
                
                bg_col = Config.COLOR_ROOM_BG
                if room.visited: bg_col = Config.COLOR_ROOM_VISITED
                if room.seq_id == game.get_current_room_seq_id(): bg_col = (85, 95, 115)
                
                pygame.draw.rect(screen, bg_col, cell_rect)
                pygame.draw.rect(screen, (60, 60, 70), cell_rect, 1)

    # 2. 墙壁
    wall_thickness = 4
    wall_color = Config.COLOR_ROOM_WALL
    
    for rid, room in game.rooms.items():
        lx, ly = room.logical_pos
        rx = center_off_x + lx * 3 * Config.GRID_SIZE
        ry = center_off_y + ly * 3 * Config.GRID_SIZE
        size = 3 * Config.GRID_SIZE 
        
        tl = (rx, ry)
        tr = (rx + size, ry)
        bl = (rx, ry + size)
        br = (rx + size, ry + size)
        
        s = Config.GRID_SIZE
        
        if 'north' in room.doors:
            pygame.draw.line(screen, wall_color, tl, (rx + s, ry), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + 2*s, ry), tr, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tl, tr, wall_thickness)
            
        if 'south' in room.doors:
            pygame.draw.line(screen, wall_color, bl, (rx + s, ry + size), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + 2*s, ry + size), br, wall_thickness)
        else: pygame.draw.line(screen, wall_color, bl, br, wall_thickness)
            
        if 'west' in room.doors:
            pygame.draw.line(screen, wall_color, tl, (rx, ry + s), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx, ry + 2*s), bl, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tl, bl, wall_thickness)
            
        if 'east' in room.doors:
            pygame.draw.line(screen, wall_color, tr, (rx + size, ry + s), wall_thickness)
            pygame.draw.line(screen, wall_color, (rx + size, ry + 2*s), br, wall_thickness)
        else: pygame.draw.line(screen, wall_color, tr, br, wall_thickness)

        id_surf = font_sm.render(str(room.seq_id), True, (100, 110, 130))
        screen.blit(id_surf, (rx + 5, ry + 5))

        if room.fragment:
            cx = int(rx + 1.5 * s)
            cy = int(ry + 1.5 * s)
            
            # [修改] 尝试使用图片绘制地图上的物品
            img_size = int(Config.GRID_SIZE * 0.8) # 稍微小一点适应格子
            img = asset_manager.get_image(room.fragment, img_size, img_size)
            
            if img:
                screen.blit(img, (cx - img_size // 2, cy - img_size // 2))
            else:
                # 兜底绘制逻辑
                f_col = Config.FRAGMENT_COLORS.get(room.fragment, (200,200,200))
                pygame.draw.circle(screen, f_col, (cx, cy), s//2 - 4)
                ft = font_sm.render(room.fragment, True, (0,0,0))
                screen.blit(ft, (cx-ft.get_width()//2, cy-ft.get_height()//2))

    # 3. 玩家
    p_sx = center_off_x + game.player_x * Config.GRID_SIZE
    p_sy = center_off_y + game.player_y * Config.GRID_SIZE
    pygame.draw.rect(screen, Config.COLOR_PLAYER, (p_sx+4, p_sy+4, Config.GRID_SIZE-8, Config.GRID_SIZE-8), border_radius=3)

    # --- 右侧面板 ---
    draw_panel(screen, panel_right, "背包 (2格合成)")
    bp_start_y = panel_right.y + 60
    game.backpack_rects = {}
    game.drop_rects = {}
    
    for i in range(Config.BACKPACK_CAPACITY):
        col, row = i % 2, i // 2
        bx, by = panel_right.x + 50 + col * 100, bp_start_y + row * 100
        rect = pygame.Rect(bx, by, 80, 80)
        game.backpack_rects[i] = rect
        
        color = (50, 56, 70)
        if i in game.selected_items: color = Config.COLOR_HIGHLIGHT
        pygame.draw.rect(screen, color, rect, border_radius=8)
        pygame.draw.rect(screen, Config.COLOR_BORDER, rect, 2, border_radius=8)
        
        if i < len(game.backpack):
            item = game.backpack[i]
            
            # [修改] 尝试使用图片绘制背包物品
            img_size_bp = 50
            img = asset_manager.get_image(item, img_size_bp, img_size_bp)
            
            if img:
                 screen.blit(img, (bx + 40 - img_size_bp // 2, by + 40 - img_size_bp // 2 - 10))
                 # 图片下方显示文字标签
                 ts = font_sm.render(item, True, (200, 200, 200))
                 screen.blit(ts, (bx+40-ts.get_width()//2, by+60))
            else:
                # 兜底绘制逻辑
                ic = Config.FRAGMENT_COLORS.get(item, (200,200,200))
                pygame.draw.rect(screen, ic, (bx+15, by+15, 50, 30), border_radius=4)
                ts = font_sm.render(item, True, (0,0,0))
                screen.blit(ts, (bx+40-ts.get_width()//2, by+20))
            
            dr = pygame.Rect(bx+45, by+50, 30, 20)
            if img: dr = pygame.Rect(bx+55, by+55, 20, 20) # 调整丢弃按钮位置
                
            game.drop_rects[i] = dr
            pygame.draw.rect(screen, (150, 80, 80), dr, border_radius=3)
            dt = font_sm.render("x", True, (255,255,255))
            screen.blit(dt, (dr.centerx-dt.get_width()//2, dr.centery-dt.get_height()//2))

    btn_rect = pygame.Rect(panel_right.x + 30, bp_start_y + 220, 340, 50)
    can_syn = (len(game.selected_items) == 2)
    pygame.draw.rect(screen, Config.COLOR_SUCCESS if can_syn else (70,70,80), btn_rect, border_radius=8)
    btn_txt = font_md.render("合成选中 (And)", True, (255,255,255))
    screen.blit(btn_txt, (btn_rect.centerx-btn_txt.get_width()//2, btn_rect.centery-btn_txt.get_height()//2))
    game.synth_button_rect = btn_rect
    
    ly = bp_start_y + 300
    for log in list(game.logs)[::-1]:
        screen.blit(font_sm.render(log, True, Config.COLOR_TEXT_DIM), (panel_right.x+20, ly))
        ly += 20

    if game.game_over:
        ov = pygame.Surface((Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT), pygame.SRCALPHA)
        ov.fill((0,0,0,180))
        screen.blit(ov, (0,0))
        txt = font_xl.render(game.win_reason, True, Config.COLOR_SUCCESS)
        screen.blit(txt, (Config.SCREEN_WIDTH//2 - txt.get_width()//2, Config.SCREEN_HEIGHT//2))

def draw_panel(screen, rect, title):
    pygame.draw.rect(screen, Config.COLOR_PANEL, rect, border_radius=15)
    pygame.draw.rect(screen, Config.COLOR_BORDER, rect, 2, border_radius=15)
    screen.blit(font_lg.render(title, True, Config.COLOR_TEXT_MAIN), (rect.x + 20, rect.y + 15))

def main():
    recorder = RLDataRecorder("RL_Auto_Seq_User")
    game = Game(recorder, map_type="Barbell")
    clock = pygame.time.Clock()
    running = True
    pygame.key.set_repeat(200, 50) 
    
    try:
        while running:
            mouse_pos = pygame.mouse.get_pos()
            for event in pygame.event.get():
                if event.type == pygame.QUIT: running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_1: game.reset("Barbell")
                    elif event.key == pygame.K_2: game.reset("Grid")
                    elif event.key == pygame.K_3: game.reset("Path")
                    elif event.key == pygame.K_4: game.reset("Ladder")
                    elif not game.game_over:
                        if event.key == pygame.K_f: game.pickup()
                        elif event.key == pygame.K_r: game.reset()
                        elif event.key in [pygame.K_w, pygame.K_UP]: game.move("north")
                        elif event.key in [pygame.K_s, pygame.K_DOWN]: game.move("south")
                        elif event.key in [pygame.K_a, pygame.K_LEFT]: game.move("west")
                        elif event.key in [pygame.K_d, pygame.K_RIGHT]: game.move("east")
                    elif game.game_over: game.reset()

                if event.type == pygame.MOUSEBUTTONDOWN:
                    if game.game_over: game.reset(); continue
                    handled = False
                    for i, r in game.drop_rects.items():
                        if r.collidepoint(event.pos): game.drop_item(i); handled = True; break
                    if not handled:
                        for i, r in game.backpack_rects.items():
                            if r.collidepoint(event.pos): game.toggle_select(i); handled = True; break
                    if not handled and game.synth_button_rect.collidepoint(event.pos):
                        game.attempt_synthesis(); handled = True

            draw_ui(game, screen, mouse_pos)
            pygame.display.flip()
            clock.tick(60)

    except KeyboardInterrupt: pass
    finally:
        game.recorder.save_to_file()
        pygame.quit()
        sys.exit()

if __name__ == "__main__":
    main()

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


pygame 2.5.2 (SDL 2.32.56, Python 3.11.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
正在从 D:\桌面文件夹\华东师范\Minecraft8.0\assets 加载资源...
  -> 已加载: e1.png
  -> 已加载: e2.png
  -> 已加载: e3.png
  -> 已加载: e4.png
  -> 已加载: e5.png
  -> 已加载: e6.png
  -> 已加载: b1.png
  -> 已加载: b2.png
  -> 已加载: c1.png
  -> 已加载: c2.png
  -> 已加载: d1.png
  -> 已加载: d2.png
Data Recorder Initialized. Output directory: rl_data/20251221_102000
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251221_102000/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251221_102000/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251221_102000/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251221_102000/game_log_RL_Auto_Seq_User.csv
Excel error: No module named 'openpyxl'
Data saved to CSV: rl_data/20251221_102000/game_log_RL_Auto_Seq_User.csv
E

SystemExit: 

d:\ANACONDA\envs\yh311_G\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
